In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
folder_path = '/content/drive/My Drive/IndianCurrencyDataset'

In [20]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
import os
import shutil
from PIL import Image
import numpy as np

In [21]:
# Create directories if they don't exist
train = os.path.join(folder_path, 'train')
validation = os.path.join(folder_path, 'validation')
test = os.path.join(folder_path, 'test')
os.makedirs(train, exist_ok=True)
os.makedirs(validation, exist_ok=True)
os.makedirs(test, exist_ok=True)

In [22]:
# Create class directories
train_fake = os.path.join(train, 'fake')
train_real = os.path.join(train, 'real')
val_fake = os.path.join(validation, 'fake')
val_real = os.path.join(validation, 'real')
os.makedirs(train_fake, exist_ok=True)
os.makedirs(train_real, exist_ok=True)
os.makedirs(val_fake, exist_ok=True)
os.makedirs(val_real, exist_ok=True)

In [23]:
# Get lists of training images
train_fake_images = [os.path.join(train_fake, img) for img in os.listdir(train_fake) if os.path.isfile(os.path.join(train_fake, img))]
train_real_images = [os.path.join(train_real, img) for img in os.listdir(train_real) if os.path.isfile(os.path.join(train_real, img))]

In [24]:
# Split into train and validation sets
train_fake_imgs, val_fake_imgs = train_test_split(train_fake_images, test_size=0.15, random_state=42)
train_real_imgs, val_real_imgs = train_test_split(train_real_images, test_size=0.15, random_state=42)

In [25]:
print(f"Moving {len(val_fake_imgs)} fake images to validation")
print(f"Moving {len(val_real_imgs)} real images to validation")
# Move validation images to their respective directories
for img_path in val_fake_imgs:
    dest_path = os.path.join(val_fake, os.path.basename(img_path))
    try:
        shutil.copy(img_path, dest_path)  # Use copy instead of move to be safer
        # Uncomment the line below if you want to remove from train after copying
        # os.remove(img_path)
    except Exception as e:
        print(f"Error copying {img_path}: {e}")

for img_path in val_real_imgs:
    dest_path = os.path.join(val_real, os.path.basename(img_path))
    try:
        shutil.copy(img_path, dest_path)  # Use copy instead of move to be safer
        # Uncomment the line below if you want to remove from train after copying
        # os.remove(img_path)
    except Exception as e:
        print(f"Error copying {img_path}: {e}")

Moving 20 fake images to validation
Moving 20 real images to validation


In [26]:
# Calculate remaining train images
train_images = train_fake_imgs + train_real_imgs
val_images = val_fake_imgs + val_real_imgs

print(f'Train set size: {len(train_images)}')
print(f'Validation set size: {len(val_images)}')

# Check directory contents
print("Train directory contents:", os.listdir(train))
print("Validation directory contents:", os.listdir(validation))
print("Test directory contents:", os.listdir(test))

Train set size: 218
Validation set size: 40
Train directory contents: ['real', 'fake']
Validation directory contents: ['real', 'fake']
Test directory contents: ['fake', 'real']


In [27]:
# Count images in validation directory
def count_images_in_directory(directory):
    image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.gif']
    count = 0

    if not os.path.exists(directory):
        print(f"Directory {directory} does not exist!")
        return count

    for root, _, files in os.walk(directory):
        for file in files:
            if any(file.lower().endswith(ext) for ext in image_extensions):
                count += 1

    return count

print(f"Total images in validation directory: {count_images_in_directory(validation)}")
for class_dir in os.listdir(validation):
    class_path = os.path.join(validation, class_dir)
    if os.path.isdir(class_path):
        print(f"Images in class '{class_dir}': {count_images_in_directory(class_path)}")

Total images in validation directory: 100
Images in class 'real': 49
Images in class 'fake': 51


In [28]:

# Create data generators
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

validation_datagen = ImageDataGenerator(rescale=1./255)  # No augmentation for validation
test_datagen = ImageDataGenerator(rescale=1./255)  # No augmentation for test

# Set up parameters
batch_size = 32
target_size = (224, 224)

# Create data generators
train_generator = train_datagen.flow_from_directory(
    train,
    target_size=target_size,
    batch_size=batch_size,
    class_mode='binary',
    shuffle=True,
    seed=42
)

validation_generator = validation_datagen.flow_from_directory(
    validation,
    target_size=target_size,
    batch_size=batch_size,
    class_mode='binary',
    shuffle=False,
    seed=42
)

test_generator = test_datagen.flow_from_directory(
    test,
    target_size=target_size,
    batch_size=batch_size,
    class_mode='binary',
    shuffle=False,
    seed=42
)

# Create model
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1, activation='sigmoid')
])


Found 257 images belonging to 2 classes.
Found 100 images belonging to 2 classes.
Found 210 images belonging to 2 classes.


In [58]:
# Compile model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Check model summary
model.summary()

# Train model
num_epochs = 25
history = model.fit(
    train_generator,
    epochs=num_epochs,
    validation_data=validation_generator,
    verbose=1
)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Epoch 1/25
9/9 ━━━━━━━━━━━━━━━━━━━━ 26s 2s/step - accuracy: 0.8338 - loss: 0.3760 - val_accuracy: 0.9100 - val_loss: 0.2329
Epoch 2/25
9/9 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.8791 - loss: 0.3392 - val_accuracy: 0.9100 - val_loss: 0.2273
Epoch 3/25
9/9 ━━━━━━━━━━━━━━━━━━━━ 12s 1s/step - accuracy: 0.8721 - loss: 0.3206 - val_accuracy: 0.9400 - val_loss: 0.2176
Epoch 4/25
9/9 ━━━━━━━━━━━━━━━━━━━━ 12s 1s/step - accuracy: 0.8253 - loss: 0.4047 - val_accuracy: 0.9400 - val_loss: 0.2137
Epoch 5/25
9/9 ━━━━━━━━━━━━━━━━━━━━ 9s 1s/step - accuracy: 0.8304 - loss: 0.3851 - val_accuracy: 0.9500 - val_loss: 0.2179
Epoch 6/25
9/9 ━━━━━━━━━━━━━━━━━━━━ 11s 1s/step - accuracy: 0.8350 - loss: 0.3575 - val_accuracy: 0.9400 - val_loss: 0.2131
Epoch 7/25
9/9 ━━━━━━━━━━━━━━━━━━━━ 11s 1s/step - accuracy: 0.8256 - loss: 0.3887 - val_accuracy: 0.9300 - val_loss: 0.2277
Epoch 8/25
9/9 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.8981 - loss: 0.3036 - val_accuracy: 0.9300 - val_loss: 0.2189
Epoch 9/2

In [30]:
def remove_corrupt_images(directory):
    """Identifies and removes corrupt image files from a directory."""
    import os
    from PIL import Image

    corrupt_images = []

    # Walk through all files in the directory
    for root, _, files in os.walk(directory):
        for file in files:
            file_path = os.path.join(root, file)
            try:
                # Try to open the image
                with Image.open(file_path) as img:
                    # Verify the image by trying to load it
                    img.verify()
            except (IOError, SyntaxError, ValueError) as e:
                print(f"Corrupt image found: {file_path}")
                print(f"Error: {e}")
                corrupt_images.append(file_path)

    # Remove corrupt images
    for img_path in corrupt_images:
        try:
            os.remove(img_path)
            print(f"Removed corrupt file: {img_path}")
        except Exception as e:
            print(f"Error removing {img_path}: {e}")

    return len(corrupt_images)

In [31]:
# Clean test directory
test_dir = '/content/drive/My Drive/IndianCurrencyDataset/test'
corrupted_count = remove_corrupt_images(test_dir)
print(f"Removed {corrupted_count} corrupt images from test directory")

# Also clean train and validation directories to be safe
train_dir = '/content/drive/My Drive/IndianCurrencyDataset/train'
corrupted_count = remove_corrupt_images(train_dir)
print(f"Removed {corrupted_count} corrupt images from train directory")

validation_dir = '/content/drive/My Drive/IndianCurrencyDataset/validation'
corrupted_count = remove_corrupt_images(validation_dir)
print(f"Removed {corrupted_count} corrupt images from validation directory")

Removed 0 corrupt images from test directory
Removed 0 corrupt images from train directory
Removed 0 corrupt images from validation directory


In [32]:
# Recreate data generators after cleaning directories
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    shuffle=True
)

validation_generator = validation_datagen.flow_from_directory(
    validation_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

# Verify the generators have found images
print(f"Found {train_generator.samples} training images")
print(f"Found {validation_generator.samples} validation images")
print(f"Found {test_generator.samples} test images")

Found 257 images belonging to 2 classes.
Found 100 images belonging to 2 classes.
Found 210 images belonging to 2 classes.
Found 257 training images
Found 100 validation images
Found 210 test images


In [61]:
# Evaluate on test data
test_loss, test_accuracy = model.evaluate(test_generator)
print(f'Test Loss: {test_loss}, Test Accuracy: {test_accuracy}')

7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 387ms/step - accuracy: 0.8539 - loss: 0.3019
Test Loss: 0.3174798786640167, Test Accuracy: 0.8714285492897034


In [35]:
def load_and_preprocess_image(image_path, target_size=(224, 224)):
    img = Image.open(image_path)
    img = img.resize(target_size)
    img_array = np.array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    return img_array

In [39]:
def predict_currency(image_path, model):
    preprocessed_image = load_and_preprocess_image(image_path)
    prediction = model.predict(preprocessed_image)
    if prediction[0][0] >= 0.3:
       return "Real Currency"
    else:
       return "Fake Currency"

In [59]:
image_path = '/content/drive/My Drive/IndianCurrencyDataset/2000.jpg'
result = predict_currency(image_path, model)
print(f"The predicted result is: {result}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
The predicted result is: Fake Currency


In [64]:
# Save the trained model for deployment
model.save("final-fake-currency.keras")
print("Model saved as Fake-currency.keras")

Model saved as Fake-currency.keras
